# Ensemble Evaluation: RoBERTa + Granite on QEvasion Test

Evaluates the RoBERTa (binary DNR) + Granite (3-class) ensemble on the QEvasion test set (308 samples).

**Requirements:**
- GPU runtime (for Granite 8B inference)
- Add `clear-non-reply-preds` dataset with `clear-non-reply-predictions-roberta.csv`
- Add `granite-checkpoint64` dataset with the adapter weights

**Outputs:**
- Granite standalone metrics
- RoBERTa standalone metrics  
- Ensemble metrics
- Saves `granite_qevasion_predictions.txt` for reuse

In [ ]:
!pip -q install transformers datasets accelerate peft pandas scikit-learn

In [ ]:
import os
import subprocess
from pathlib import Path

# Environment detection
IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_KAGGLE:
    WORK_DIR = Path('/kaggle/working')
    INPUT_DIR = Path('/kaggle/input')
    print('Running on Kaggle')
elif IN_COLAB:
    WORK_DIR = Path('/content')
    INPUT_DIR = WORK_DIR
    print('Running on Colab')
else:
    WORK_DIR = Path.cwd()
    INPUT_DIR = WORK_DIR
    print('Running locally')

os.chdir(WORK_DIR)
print(f'Working dir: {WORK_DIR}')

In [ ]:
# Clone repo
REPO_URL = 'https://github.com/gigibot-tech/CLARITY-SemEval-2026.git'
REPO_DIR = WORK_DIR / 'CLARITY-SemEval-2026'

if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL}...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repo exists, pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'Changed to: {REPO_DIR}')

In [ ]:
# Link input data
import shutil

# RoBERTa predictions CSV
roberta_csv_candidates = [
    INPUT_DIR / 'clear-non-reply-preds' / 'clear-non-reply-predictions-roberta.csv',
    INPUT_DIR / 'roberta-predictions' / 'clear-non-reply-predictions-roberta.csv',
    WORK_DIR / 'clear-non-reply-predictions-roberta.csv',
]
roberta_csv_dst = REPO_DIR / 'clear-non-reply-predictions-roberta.csv'

for src in roberta_csv_candidates:
    if src.exists():
        if not roberta_csv_dst.exists():
            shutil.copy(src, roberta_csv_dst)
        print(f'RoBERTa CSV: {roberta_csv_dst}')
        break
else:
    print('WARNING: RoBERTa predictions CSV not found. Add it as a Kaggle dataset.')

# Granite checkpoint
checkpoint_candidates = [
    INPUT_DIR / 'granite-checkpoint64',
    INPUT_DIR / 'checkpoint64',
    INPUT_DIR / 'granite-lora-checkpoint',
]
checkpoint_dst = REPO_DIR / 'checkpoint64'

for src in checkpoint_candidates:
    if src.exists() and (src / 'adapter_model.safetensors').exists():
        if not checkpoint_dst.exists():
            shutil.copytree(src, checkpoint_dst)
        print(f'Granite checkpoint: {checkpoint_dst}')
        break
else:
    print('WARNING: Granite checkpoint not found. Add it as a Kaggle dataset.')

# List what we have
print('\nFiles in repo root:')
for f in sorted(REPO_DIR.iterdir()):
    if f.is_file() and f.suffix in ['.csv', '.txt', '.py']:
        print(f'  {f.name}')

In [ ]:
# Run the ensemble validation script
!python scripts/run_ensemble_validation.py

In [ ]:
# Show saved results
import json

results_file = REPO_DIR / 'results' / 'ensemble_qevasion_metrics.json'
if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    print('=== Saved Results ===')
    print(json.dumps(results, indent=2))
else:
    print('Results file not found')

In [ ]:
# Copy outputs to /kaggle/working for download
if IN_KAGGLE:
    outputs = [
        REPO_DIR / 'granite_qevasion_predictions.txt',
        REPO_DIR / 'results' / 'ensemble_qevasion_metrics.json',
    ]
    for src in outputs:
        if src.exists():
            dst = WORK_DIR / src.name
            shutil.copy(src, dst)
            print(f'Copied: {dst}')